# 프로젝트 2. 자영업자 재무 분석

- 작성: 양재일 (서울사이버대학교 인공지능학과)
- 목적: 재무애로 컨설팅에서 쓰던 수익성·안정성 진단을, 가상 재무제표로 재현한다.
- 사용 도구: Python, pandas, NumPy, matplotlib
- 데이터: **가상 자료** (난수로 생성). 실제 상담 고객·상호·재무자료는 사용하지 않는다.

> 이 노트북의 가게 이름과 숫자는 모두 가명·가상입니다. 컨설팅 결과로 인용할 수 없습니다.

## 1. 문제 정의

자영업자 재무애로 상담에서 가장 먼저 하던 일은 두 가지였다.

1. **벌어들이는가?** (수익성) — 매출 대비 원가·경비·순이익
2. **버틸 수 있는가?** (안정성) — 유동성, 부채 부담, 자본 완충

상담 현장에서는 이 계산을 엑셀로 반복했다. 같은 절차를 코드로 옮기면

- 여러 업체를 같은 기준으로 비교하고
- 위험 등급을 자동으로 붙이고
- 한 업체에 대한 진단 문장을 바로 만들 수 있다.

이번 초안은 가상 자영업자 80곳의 손익·재무상태 숫자를 만든 뒤, 핵심 6개 비율과 위험 등급을 산출한다.

## 2. 데이터 (가상 재무제표)

업종은 상담에서 자주 보던 네 가지로 단순화했다. 업종별 원가율·경비율은 학습용 가정이다.

| 업종 | 매출총이익률 가정 | 특징 |
| --- | --- | --- |
| 음식점 | 중간 | 인건비·임차료 비중이 큼 |
| 소매점 | 낮음 | 매입 원가 비중이 큼 |
| 서비스 | 높음 | 원가는 낮고 고정비 영향 |
| 제조 소공인 | 중간 | 재고·운전자금 부담 |

금액 단위는 **만원**이다.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import os, sys, subprocess
from pathlib import Path
from matplotlib import font_manager as fm

def setup_korean_font():
    font_path = Path("/usr/share/fonts/truetype/nanum/NanumGothic.ttf")
    if sys.platform.startswith("linux") and not font_path.exists():
        subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-nanum"], check=False, capture_output=True)
    if font_path.exists():
        fm.fontManager.addfont(str(font_path))
        plt.rcParams["font.family"] = "NanumGothic"
    plt.rcParams["axes.unicode_minus"] = False

setup_korean_font()
plt.rcParams["figure.figsize"] = (8, 4.5)

RNG = np.random.default_rng(42)
N = 80
INDUSTRIES = np.array(["음식점", "소매점", "서비스", "제조소공인"])

GROSS_MARGIN = {"음식점": 0.62, "소매점": 0.28, "서비스": 0.72, "제조소공인": 0.38}
OPEX_RATIO = {"음식점": 0.50, "소매점": 0.18, "서비스": 0.48, "제조소공인": 0.24}

industry = RNG.choice(INDUSTRIES, size=N, p=[0.35, 0.25, 0.25, 0.15])
sales = RNG.integers(8_000, 90_000, size=N)

gross_margin = np.array([GROSS_MARGIN[i] for i in industry]) + RNG.normal(0, 0.04, N)
gross_margin = np.clip(gross_margin, 0.12, 0.85)
cogs = (sales * (1 - gross_margin)).round().astype(int)
gross_profit = sales - cogs

opex_ratio = np.array([OPEX_RATIO[i] for i in industry]) + RNG.normal(0, 0.05, N)
opex_ratio = np.clip(opex_ratio, 0.08, 0.70)
opex = (sales * opex_ratio).round().astype(int)
operating_profit = gross_profit - opex

interest = RNG.integers(50, 2_400, size=N)
net_profit = operating_profit - interest + RNG.integers(-400, 400, size=N)

current_assets = (sales * RNG.uniform(0.12, 0.45, N) + RNG.integers(500, 8_000, N)).round().astype(int)
noncurrent_assets = RNG.integers(1_000, 40_000, size=N)
total_assets = current_assets + noncurrent_assets

current_liab = (current_assets * RNG.uniform(0.35, 1.40, N)).round().astype(int)
noncurrent_liab = RNG.integers(0, 25_000, size=N)
total_liab = current_liab + noncurrent_liab
equity = np.maximum(total_assets - total_liab, 300)
total_assets = total_liab + equity

raw = pd.DataFrame({
    "업체코드": [f"VIRT-{i:03d}" for i in range(1, N + 1)],
    "업종": industry,
    "매출액": sales,
    "매출원가": cogs,
    "매출총이익": gross_profit,
    "판관비": opex,
    "영업이익": operating_profit,
    "이자비용": interest,
    "당기순이익": net_profit,
    "유동자산": current_assets,
    "비유동자산": noncurrent_assets,
    "자산총계": total_assets,
    "유동부채": current_liab,
    "비유동부채": noncurrent_liab,
    "부채총계": total_liab,
    "자본총계": equity,
})
print(raw.shape)
raw.head()

## 3. 방법 — 핵심 재무비율과 위험 등급

상담 때 쓰던 최소 지표 6개만 계산한다.

| 구분 | 지표 | 계산 |
| --- | --- | --- |
| 수익성 | 매출총이익률 | 매출총이익 ÷ 매출액 × 100 |
| 수익성 | 영업이익률 | 영업이익 ÷ 매출액 × 100 |
| 수익성 | 순이익률 | 당기순이익 ÷ 매출액 × 100 |
| 안정성 | 유동비율 | 유동자산 ÷ 유동부채 × 100 |
| 안정성 | 부채비율 | 부채총계 ÷ 자본총계 × 100 |
| 안정성 | 이자보상배율 | 영업이익 ÷ 이자비용 |

**위험 등급 규칙 (학습용 단순 기준)**  
아래 항목 중 해당하는 개수로 등급을 붙인다.

- 영업이익률 < 0
- 유동비율 < 100
- 부채비율 > 200
- 이자보상배율 < 1.5

| 해당 개수 | 등급 |
| --- | --- |
| 0 | 양호 |
| 1 | 주의 |
| 2 이상 | 위험 |

In [ ]:
def add_ratios(frame):
    out = frame.copy()
    out["매출총이익률"] = out["매출총이익"] / out["매출액"] * 100
    out["영업이익률"] = out["영업이익"] / out["매출액"] * 100
    out["순이익률"] = out["당기순이익"] / out["매출액"] * 100
    out["유동비율"] = out["유동자산"] / out["유동부채"] * 100
    out["부채비율"] = out["부채총계"] / out["자본총계"] * 100
    out["이자보상배율"] = np.where(out["이자비용"] > 0, out["영업이익"] / out["이자비용"], np.nan)

    flags = (
        (out["영업이익률"] < 0).astype(int)
        + (out["유동비율"] < 100).astype(int)
        + (out["부채비율"] > 200).astype(int)
        + (out["이자보상배율"] < 1.5).astype(int)
    )
    out["위험점수"] = flags
    out["위험등급"] = pd.cut(flags, bins=[-1, 0, 1, 10], labels=["양호", "주의", "위험"])
    return out

df = add_ratios(raw)
ratio_cols = ["매출총이익률", "영업이익률", "순이익률", "유동비율", "부채비율", "이자보상배율"]
display(df[ratio_cols + ["위험등급"]].describe().round(1))
print(df["위험등급"].value_counts())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

grade_order = ["양호", "주의", "위험"]
colors = ["#4c9a8a", "#e0a145", "#c45c4c"]
counts = df["위험등급"].value_counts().reindex(grade_order)
axes[0].bar(grade_order, counts, color=colors)
axes[0].set_title("위험 등급 분포 (가상 80곳)")
axes[0].set_ylabel("업체 수")

industry_grade = pd.crosstab(df["업종"], df["위험등급"]).reindex(columns=grade_order)
industry_grade.plot(kind="bar", stacked=True, ax=axes[1], color=colors)
axes[1].set_title("업종별 위험 등급")
axes[1].tick_params(axis="x", rotation=0)
axes[1].legend(title="등급")
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))

for ind in INDUSTRIES:
    part = df[df["업종"] == ind]
    axes[0].scatter(part["유동비율"], part["영업이익률"], alpha=0.75, label=ind)
axes[0].axvline(100, color="gray", linestyle="--", linewidth=0.8)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=0.8)
axes[0].set_xlabel("유동비율(%)")
axes[0].set_ylabel("영업이익률(%)")
axes[0].set_title("수익성 × 유동성")
axes[0].legend()

df.boxplot(column="부채비율", by="업종", ax=axes[1])
axes[1].axhline(200, color="#c45c4c", linestyle="--", linewidth=0.8)
axes[1].set_title("업종별 부채비율 (점선=200%)")
axes[1].set_xlabel("")
plt.suptitle("")
plt.tight_layout()
plt.show()

## 4. 개별 진단서 (컨설팅 절차 재현)

상담 마지막에 건네던 한 장 요약을 함수로 만든다. 가상 업체 한 곳을 골라 출력한다.

In [ ]:
def diagnose(row):
    lines = [
        f"[{row.업체코드}] {row.업종} 재무 진단 (가상 자료)",
        f"- 매출 {row.매출액:,}만원 / 영업이익 {row.영업이익:,}만원 / 순이익 {row.당기순이익:,}만원",
        f"- 수익성: 총이익률 {row.매출총이익률:.1f}%, 영업이익률 {row.영업이익률:.1f}%, 순이익률 {row.순이익률:.1f}%",
        f"- 안정성: 유동비율 {row.유동비율:.1f}%, 부채비율 {row.부채비율:.1f}%, 이자보상배율 {row.이자보상배율:.1f}배",
        f"- 종합 등급: {row.위험등급} (위험점수 {int(row.위험점수)}/4)",
        "- 실무 코멘트:",
    ]
    if row.영업이익률 < 0:
        lines.append("  · 영업손실입니다. 원가·인건비·임차료 중 먼저 줄일 항목을 정해야 합니다.")
    elif row.영업이익률 < 5:
        lines.append("  · 영업이익률이 낮습니다. 가격·객단가와 고정비를 같이 점검하십시오.")
    else:
        lines.append("  · 영업 수익성은 방어되고 있습니다.")

    if row.유동비율 < 100:
        lines.append("  · 유동비율이 100% 미만입니다. 만기 도래 부채와 재고·매출채권 회수를 우선 보십시오.")
    if row.부채비율 > 200:
        lines.append("  · 부채비율이 높습니다. 신규 차입보다 만기 분산과 자본 보강이 필요합니다.")
    if row.이자보상배율 < 1.5:
        lines.append("  · 영업이익으로 이자를 충분히 감당하기 어렵습니다. 금리·한도 조정을 검토하십시오.")
    if row.위험점수 == 0:
        lines.append("  · 당장은 긴급 신호가 없습니다. 계절 매출 변동에 대비한 현금 보유를 유지하십시오.")
    return "\n".join(lines)

sample_risk = df.sort_values("위험점수", ascending=False).iloc[0]
sample_ok = df.sort_values("위험점수").iloc[0]
print(diagnose(sample_risk))
print("\n" + "-" * 60 + "\n")
print(diagnose(sample_ok))

In [ ]:
summary = (
    df.groupby("업종")[["매출총이익률", "영업이익률", "유동비율", "부채비율", "이자보상배율"]]
    .median()
    .round(1)
)
print("업종별 중앙값")
display(summary)

print("\n위험 등급 상위 8곳 (가상)")
display(
    df.sort_values("위험점수", ascending=False)[
        ["업체코드", "업종", "매출액", "영업이익률", "유동비율", "부채비율", "이자보상배율", "위험등급"]
    ].head(8).round(1)
)

## 5. 결과 해석

난수이므로 실행할 때마다 세부 숫자는 같게 나오도록 시드(42)를 고정했다. 읽기 포인트는 다음과 같다.

1. **수익성만 보면 안 된다.** 영업이익률이 괜찮은 가게도 유동비율이 100% 미만이면 급여·임대료 시점에 돈이 막힌다.
2. **부채비율 200%를 넘으면** 자기자본보다 빚이 두 배 이상이다. 상담에서는 신규 시설 투자 자제를 먼저 권했다.
3. **이자보상배율 1.5배 미만**은 영업으로 이자를 겨우 막는 상태다. 금리가 오르면 바로 적자로 전환된다.
4. 업종별로 총이익률 수준이 다르다. 소매점은 총이익률이 낮아도 회전이 빠르면 버틸 수 있고, 음식점은 고정비 때문에 매출이 조금만 줄어도 영업손실이 난다.

이 구조는 컨설팅 당시 엑셀 점검표를 코드로 옮긴 것이다. 다음 단계에서는 월별 현금흐름(입금·카드·현금·공과금)을 붙이면 더 현장에 가까워진다.

## 6. 배운 점과 한계

**배운 점**
- 손익계산서와 재무상태표 숫자만 있으면, pandas로 수익성·안정성 진단을 표준화할 수 있다.
- 위험 등급은 복잡한 머신러닝보다, 실무에서 쓰던 임계값(유동비율 100, 부채비율 200 등)을 규칙으로 두는 편이 설명하기 쉽다.
- 개별 진단 문장을 함수로 만들면 같은 기준을 여러 업체에 반복 적용할 수 있다.

**한계**
- 데이터는 가상이다. 업종 원가율은 가정이며 통계청 수치와 다를 수 있다.
- 실제 자영업 위험은 카드 정산 주기, 임대차, 보증, 가족 인건비처럼 장부에 안 잡히는 항목이 크다.
- 연간 숫자만 써서 계절성(여름 비수기 등)을 보지 못했다.

**다음 개선**
- 공공 소상공인 통계의 업종별 평균과 가상 업체의 중앙값을 비교한다.
- 월별 현금흐름표를 가상 생성해 ‘흑자도산’ 패턴을 보여 준다.
- 프로젝트 1의 신용 특성과 이 재무비율을 한 화면에 붙인다.